In [1]:
# ============================================================
#  03b H3 Hex Grid — 统一六边形网格
#  生成 250m (res 9) 和 500m (res 8) 两套 hex grid
#  供所有下游分析统一使用
# ============================================================

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import h3
import json
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
import warnings
warnings.filterwarnings("ignore")

OUT = Path("data_out")
OUT.mkdir(exist_ok=True)
BOUNDARY = OUT / "shenzhen_boundary.geojson"

shenzhen = gpd.read_file(BOUNDARY).to_crs(4326)
shenzhen_geom = shenzhen.union_all() if hasattr(shenzhen, "union_all") else shenzhen.unary_union

# 将 shapely 几何转为 GeoJSON dict
sz_json = json.loads(shenzhen.to_json())
poly_geojson = sz_json["features"][0]["geometry"]

def h3_to_polygon(h):
    coords = h3.cell_to_boundary(h)
    return Polygon([(lng, lat) for lat, lng in coords])

for res, label in [(8, "~460m"), (9, "~174m")]:
    hexes = h3.geo_to_cells(poly_geojson, res=res)
    hex_gdf = gpd.GeoDataFrame(
        {"h3_id": list(hexes)},
        geometry=[h3_to_polygon(h) for h in hexes],
        crs=4326,
    )
    outfile = OUT / f"sz_hex_grid_res{res}.gpkg"
    hex_gdf.to_file(outfile, driver="GPKG")
    print(f"H3 res {res} ({label}): {len(hex_gdf):,} hexagons -> {outfile}")

print("\nDone.")

H3 res 8 (~460m): 2,724 hexagons -> data_out/sz_hex_grid_res8.gpkg
H3 res 9 (~174m): 19,063 hexagons -> data_out/sz_hex_grid_res9.gpkg

Done.
